In [2]:
import numpy as np
import pandas as pd 
import matplotlib as plt
from collections import Counter


In [3]:
data = pd.read_csv("Iris.csv")
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
x = data.drop(columns=["Id", "Species"]).values
y = data['Species'].values

In [7]:
len(x), len(y)

(150, 150)

In [13]:
l = len(x) - 1
x[l:]

array([[5.9, 3. , 5.1, 1.8]])

In [14]:
# train test split
splitX = int(len(x)*0.8) # 80-20 % split
splitY = int(len(y)*0.8)
x_train, x_test = x[:splitX], x[splitX:]
y_train, y_test = y[:splitY], y[splitY:]

In [15]:
len(x_train), len(x_test)

(120, 30)

# Decision Tree Implemenetation

In [30]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, infoGain=None, left=None, right=None, value=None):
        
        # Decision Node
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.infoGain = infoGain
        self.left = left
        self.right = right
        
        # Leaf Node
        self.value = value


In [31]:
class DecisionTree:
    def __init__(self, min_sample_split=2, max_depth=2):

        self.min_sample_split = min_sample_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if n_samples >= self.min_sample_split and curr_depth <= self.max_depth:
            best_split = self.best_split(dataset, n_features)

            if best_split["infoGain"] > 0:
                left_node = self.build_tree(best_split["left_dataset"], curr_depth+1)
                right_node = self.build_tree(best_split["right_dataset"], curr_depth+1)

                return Node(best_split["feature_idx"], best_split["threshold"], best_split["infoGain"], left_node, right_node)
            
        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)
    

    def best_split(self, dataset, n_features):
        best_split = {'feature_idx': None, 'threshold': None, 'infoGain': -1, 'left_dataset': None, 'right_dataset': None}

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                left_dataset, right_dataset = self.split(dataset, feature_idx, threshold)

                if len(left_dataset) and len(right_dataset):
                    parent_y, left_y, right_y = dataset[:, -1], left_dataset[:, -1], right_dataset[:, -1]

                    infoGain = self.information_gain(parent_y, left_y, right_y)

                    if infoGain > best_split['infoGain']:
                        best_split['feature_idx'] = feature_idx
                        best_split['threshold'] = threshold
                        best_split['infoGain'] = infoGain
                        best_split['left_dataset'] = left_dataset
                        best_split['right_dataset'] = right_dataset

        return best_split
    

    def split(self, dataset, feature_idx, threshold):
        left_dataset = np.array([row for row in dataset if row[feature_idx] <= threshold])
        right_dataset = np.array([row for row in dataset if row[feature_idx] > threshold])

        return left_dataset, right_dataset
    

    def information_gain(self, parent_y, left_y, right_y):
        left_weight = len(left_y) / len(parent_y)
        right_weight = len(right_y) / len(parent_y)

        infoGain = self.entropy(parent_y) - (left_weight * self.entropy(left_y) + right_weight * self.entropy(right_y))
        return infoGain
    

    def entropy(self, y):
        entropy = 0

        class_labels = np.unique(y)
        for class_label in class_labels:
            p = len(y[y==class_label]) / len(y)
            if p > 0:
                entropy += -p*np.log2(p)

        return entropy
    
    def fit(self, x, y):
        dataset = np.concatenate([x, y.reshape(-1, 1)], axis=1)
        self.root = self.build_tree(dataset)

    def predict(self, x):
        predictions = [self.predict_class(row, self.root) for row in x]
        return predictions
    
    def predict_class(self, row, node):
        if node.value != None:
            return node.value
        
        feature_val = row[node.feature_idx]
        if feature_val <= node.threshold:
            return self.predict_class(row, node.left)
        else:
            return self.predict_class(row, node.right)
        
    def print_tree(self, node=None, depth=0, indent="|   "):
        prefix = indent * depth

        if node is None:
            node = self.root

        if node.value is not None:
            print(f"{prefix}|--- class: {node.value}")
            return

        feature_label = f"Feature {node.feature_idx}"

        print(f"{prefix}|--- {feature_label} <= {node.threshold}")
        self.print_tree(node.left, depth + 1, indent)

        print(f"{prefix}|--- {feature_label} > {node.threshold}")
        self.print_tree(node.right, depth + 1, indent)
        


In [32]:
dt = DecisionTree(min_sample_split=2, max_depth=2)
dt.fit(x_train, y_train)
predictions = dt.predict(x_test)

accuracy = np.mean(predictions == y_test) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 73.33%


In [33]:
dt.print_tree()

|--- Feature 2 <= 1.9
|   |--- class: Iris-setosa
|--- Feature 2 > 1.9
|   |--- Feature 2 <= 4.9
|   |   |--- Feature 0 <= 4.9
|   |   |   |--- class: Iris-versicolor
|   |   |--- Feature 0 > 4.9
|   |   |   |--- class: Iris-versicolor
|   |--- Feature 2 > 4.9
|   |   |--- Feature 3 <= 1.7
|   |   |   |--- class: Iris-versicolor
|   |   |--- Feature 3 > 1.7
|   |   |   |--- class: Iris-virginica
